# 🔗 3.4 LCEL and Runnables

## Learning Objectives
In this notebook, you will learn:
1. **Manual vs. LCEL pipelines** - build the same prompt → model → parser flow by hand, then again with the `|` operator, to see exactly what LCEL saves you
2. **The Runnable interface's core methods** - `.invoke()`, `.stream()`, `.batch()`, and their async counterparts `.ainvoke()`, `.astream()`, `.abatch()`
3. **Retrieval-Augmented Generation (RAG) with LCEL** - wire a Chroma retriever into a prompt using both `itemgetter` and `RunnablePassthrough`
4. **Two equivalent ways to shape chain input** - a dict of `{"context": ..., "question": ...}` built explicitly vs. built with `RunnablePassthrough()`

## Prerequisites
- Completion of `3.1_LCEL_Introduction.ipynb`, `3.2_Runnables.ipynb`, and `3.3_LCEL_Deepdive.ipynb`
- An `OPENAI_API_KEY` set in a `.env` file at the project root
- `langchain-chroma` installed (used here for a small in-memory vector store)

> **Source**: This notebook's exercises are adapted from the Udemy course [LangChain in Action](https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/).

Link: https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and load .env
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================================
# NON-LCEL BASELINE: Build a prompt template
# ============================================================================
prompt = ChatPromptTemplate.from_template("Tell me an interesting fact about {topic}")

In [ ]:
# ============================================================================
# NON-LCEL BASELINE: Format the prompt manually with .invoke()
# ============================================================================
prompt_val = prompt.invoke({"topic": "dog"})
print(prompt_val)

In [ ]:
# ============================================================================
# NON-LCEL BASELINE: Inspect the formatted messages
# ============================================================================
print(prompt_val.to_messages())

In [ ]:
# ============================================================================
# NON-LCEL BASELINE: Pass the formatted prompt to the model manually
# ============================================================================
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")
result = model.invoke(prompt_val)
result

In [ ]:
# ============================================================================
# NON-LCEL BASELINE: Parse the result manually
# ============================================================================
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(result)

### Now let´s do this LCEL

In [ ]:
# ============================================================================
# LCEL CHAIN: Build the same pipeline with the | operator
# ============================================================================
prompt = ChatPromptTemplate.from_template("Tell me an interesting fact about {topic}")
model = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

basicchain = model | output_parser

In [ ]:
# ============================================================================
# LCEL CHAIN: Invoke the model+parser chain directly with a raw string
# ============================================================================
basicchain.invoke("hello!")

In [ ]:
# ============================================================================
# LCEL CHAIN: Full chain — prompt | model | output_parser
# ============================================================================
chain = prompt | model | output_parser

chain.invoke({"topic": "dog"})

### Most important methods of the Runnable Interface (Stream, Invoke, Batch)

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .stream() — token-by-token output
# ============================================================================
for s in chain.stream({"topic": "bears"}):
    print(s, end="", flush=True)

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .invoke() — synchronous single call
# ============================================================================
chain.invoke({"topic": "bears"})

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .batch() — process multiple inputs at once
# ============================================================================
chain.batch([{"topic": "bears"}, {"topic": "cats"}])

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .astream() — async token streaming
# ============================================================================
async for s in chain.astream({"topic": "bears"}):
    print(s, end="", flush=True)

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .ainvoke() — async single call
# ============================================================================
await chain.ainvoke({"topic": "bears"})

In [ ]:
# ============================================================================
# RUNNABLE INTERFACE: .abatch() — async batch processing
# ============================================================================
await chain.abatch([{"topic": "bears"}, {"topic": "cats"}])

### Retrieval Augmented Generation with LCEL

In [ ]:
# ============================================================================
# RAG SETUP: Build a tiny in-memory vector store with Chroma
# ============================================================================
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

embedding_function = OpenAIEmbeddings()

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embedding_function)
retriever = db.as_retriever()

In [ ]:
# ============================================================================
# RAG SETUP: Query the retriever directly (old method reference)
# ============================================================================
retriever.invoke("What does the dog want to eat?")  # old method

In [ ]:
# ============================================================================
# RAG SETUP: Query the retriever directly
# ============================================================================
retriever.invoke("What does the dog want to eat?")

In [ ]:
# ============================================================================
# RAG CHAIN: Define the context+question prompt template
# ============================================================================
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# ============================================================================
# RAG CHAIN: Wire the retriever and question into the prompt (itemgetter form)
# ============================================================================
from operator import itemgetter

retrieval_chain = (
    {
        "context": (lambda x: x["question"]) | retriever,
        # "question": lambda x: x["question"],
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
# ============================================================================
# RAG CHAIN: Invoke with a dict input
# ============================================================================
retrieval_chain.invoke({"question": "What does the dog like to eat?"})

In [ ]:
# ============================================================================
# RAG CHAIN: Simpler wiring with RunnablePassthrough for the question
# ============================================================================
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
# ============================================================================
# RAG CHAIN: Invoke with a plain string input
# ============================================================================
retrieval_chain.invoke("What does the dog like to eat?")

---
## 📝 Summary

In this notebook, we learned:

### 1. Manual vs. LCEL Pipelines
- Doing it by hand means three separate calls: `prompt.invoke(...)` → `model.invoke(prompt_val)` → `output_parser.invoke(result)`
- LCEL collapses that into `chain = prompt | model | output_parser`, then a single `chain.invoke({...})`

### 2. The Runnable Interface
- Every LCEL chain exposes the same methods regardless of what it's built from:

| Method | Behavior |
|--------|----------|
| `.invoke()` | Run once, synchronously |
| `.stream()` | Yield output incrementally |
| `.batch()` | Process a list of inputs |
| `.ainvoke()` / `.astream()` / `.abatch()` | Async equivalents |

### 3. Retrieval-Augmented Generation with LCEL
- A `Chroma` vector store's `.as_retriever()` is itself a `Runnable`, so it composes directly into a chain: `{"context": retriever, "question": ...} | prompt | model | parser`
- **`itemgetter("question")`** and **`RunnablePassthrough()`** are two equivalent ways to route the same input value into a parallel branch — `itemgetter` when you need to pull one key out of a larger dict, `RunnablePassthrough` when the whole input *is* that value

### Key Pattern
```python
retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)
```

### Next Steps
- Continue to **`3.5_Chain_Migrations.ipynb`** and **`3.6_Chain_Migration_Advanced.ipynb`** to see how legacy chain classes map onto these same Runnable primitives
- Then **`3.7_Branching_Routing_Merging_Chains.ipynb`** for the LCEL-native equivalents of branching, routing, and merging
- Revisit **`3.3_LCEL_Deepdive.ipynb`** if any of `RunnableParallel`, `.assign()`, or the `|` operator's mechanics need a refresher
- `03_Legacy_Chains/` covers the same chain-building ground with the pre-1.x `LLMChain`-family API, kept for historical/migration context
